# STT API — Colab 실행 노트북

자체 호스팅과 **동일한 코드**를 Google Colab(Pro)에서 실행한다.
저장소를 클론해 의존성을 설치하고, GPU에서 모델을 프리로드한 뒤 API 서버를 띄운다.

## 준비
1. 런타임 → 런타임 유형 변경 → **GPU** 선택 (T4 이상)
2. 아래 셀을 위에서부터 실행
3. 외부에서 접속하려면 cloudflared 임시 터널의 공개 URL을 사용 (아래 출력)

> Colab은 데모/개발용이다. 세션 종료 시 서버가 멈추고, 터널 URL은 공개된다.
> 운영은 `run.sh`로 자체 호스팅하고, 공개 URL에는 반드시 `STT_API_KEY`를 설정한다.

In [ ]:
!nvidia-smi || echo 'GPU 없음 - CPU로 실행됩니다(느림)'
import platform, sys
print(platform.platform(), sys.version.split()[0])

MODEL = 'small'   # T4: small / L4·A100: 'large-v3-turbo'
PORT = 8000
print('model:', MODEL)

In [ ]:
import os, shutil, subprocess

# 셀을 여러 번 실행해도 같은 위치에서 시작한다 (중첩 chdir 방지)
WORK_ROOT = '/content' if os.path.isdir('/content') else os.getcwd()
os.chdir(WORK_ROOT)

REPO_URL = 'https://github.com/lukehemmin/STT_API.git'
REPO_DIR = 'STT_API'
REQUIRED = ['requirements.txt', 'app/main.py', 'app/engine.py']
REPO_PATH = os.path.join(WORK_ROOT, REPO_DIR)

def clone():
    shutil.rmtree(REPO_PATH, ignore_errors=True)
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_PATH], check=True)

if not os.path.isdir(os.path.join(REPO_PATH, '.git')):
    clone()
else:
    subprocess.run(['git', '-C', REPO_PATH, 'pull', '--ff-only'], check=False)
    if not os.path.exists(os.path.join(REPO_PATH, 'requirements.txt')):
        print('기존 클론이 오래되어 새로 클론합니다.')
        clone()

os.chdir(REPO_PATH)
missing = [p for p in REQUIRED if not os.path.exists(p)]
if missing:
    raise RuntimeError('클론한 저장소에 코드가 없습니다: ' + ', '.join(missing))
print('cwd:', os.getcwd())
print('HEAD:', subprocess.run(['git', 'rev-parse', '--short', 'HEAD'],
                               capture_output=True, text=True).stdout.strip())


In [ ]:
import shutil, subprocess, sys

def run(cmd):
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        print('실패:', ' '.join(cmd))
        print((res.stderr or res.stdout)[-4000:])
        raise RuntimeError('명령 실패: ' + ' '.join(cmd))
    return res

subprocess.run(['apt-get', '-qq', 'update'], capture_output=True)  # 경고만 나오는 경우가 있어 무시
run(['apt-get', '-qq', 'install', '-y', 'ffmpeg'])
run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])

try:
    run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
except RuntimeError:
    print()
    print('[힌트] Python ' + sys.version.split()[0] + ' 에서 의존성 설치가 실패했습니다.')
    print('  - 이 Python 버전용 휠이 없을 수 있습니다. 최신 빌드를 시도해 보세요:')
    print('      pip install -U ctranslate2 faster-whisper')
    print('  - 그래도 실패하면 런타임 → 런타임 유형 변경에서 Python 3.11 로 바꿔 다시 실행하세요.')
    raise

import importlib.metadata as md
for pkg in ('faster-whisper', 'ctranslate2', 'av'):
    try:
        print(pkg, md.version(pkg))
    except Exception:
        print(pkg, '(버전 확인 실패)')

# PyAV 19부터 av.open()의 metadata_errors 인자가 사라졌는데 faster-whisper 1.2.x는
# 여전히 그 인자를 넘긴다. 이 조합이면 배치 전사가 전부 TypeError로 실패한다.
av_version = md.version('av')
if int(av_version.split('.')[0]) >= 19:
    print('av', av_version, '는 faster-whisper와 호환되지 않습니다. 19 미만으로 내립니다.')
    run([sys.executable, '-m', 'pip', 'install', '-q', 'av>=11,<19'])
    print('av (설치 후):', md.version('av'))

# ctranslate2 4.9는 CUDA 12용 cuBLAS/cuDNN을 지연 로드한다. CUDA 13 이미지처럼
# libcublas.so.12 가 없으면 모델 로드는 성공하고 첫 추론에서만 실패한다.
import ctypes, os, site

def has_lib(name):
    try:
        ctypes.CDLL(name)
        return True
    except OSError:
        return False

if not (has_lib('libcublas.so.12') and has_lib('libcudnn.so.9')):
    print('CUDA 12용 cuBLAS/cuDNN이 없습니다. pip 휠로 설치합니다(수백 MB).')
    run([sys.executable, '-m', 'pip', 'install', '-q',
         'nvidia-cublas-cu12', 'nvidia-cudnn-cu12==9.*'])

nvidia_libs = [os.path.join(site.getsitepackages()[0], 'nvidia', part, 'lib')
               for part in ('cublas', 'cudnn')]
nvidia_libs = [d for d in nvidia_libs if os.path.isdir(d)]
if nvidia_libs:
    os.environ['LD_LIBRARY_PATH'] = ':'.join(
        nvidia_libs + [os.environ.get('LD_LIBRARY_PATH', '')]).strip(':')
    print('LD_LIBRARY_PATH:', os.environ['LD_LIBRARY_PATH'])
print('cublas12(현재 프로세스):', has_lib('libcublas.so.12'),
      '| cudnn9:', has_lib('libcudnn.so.9'))
print('  False여도 프리로드/서버 서브프로세스에는 LD_LIBRARY_PATH가 적용된다')

import ctranslate2
print('CUDA device count:', ctranslate2.get_cuda_device_count())
print('ffmpeg:', shutil.which('ffmpeg'))
print('의존성 설치 완료')


In [ ]:
import os, subprocess

res = subprocess.run(['python', '-m', 'app.preload'], capture_output=True, text=True,
                     env=dict(os.environ, STT_MODEL=MODEL))
print(res.stdout.strip())

if res.returncode != 0:
    err = res.stderr or ''
    print(err[-4000:])
    if any(k in err.lower() for k in ('cublas', 'cudnn', 'libcu', 'no kernel image')):
        print('\n[힌트] GPU용 CUDA 12 라이브러리 문제입니다. 셀 3을 다시 실행하면 자동으로 설치합니다.')
        print('      그다음 이 셀과 셀 5(서버)를 다시 실행하세요. 셀 3만 다시 실행하면 옛 서버는 그대로입니다.')
        print("      CPU로 우회: 셀 5의 env 에 STT_DEVICE='cpu' 추가 (매우 느림)")
    raise RuntimeError('모델 프리로드 실패')
print('프리로드 완료:', MODEL)


In [ ]:
import os, re, subprocess, time, requests

# 재실행 시 이전 서버/터널을 먼저 정리한다. 옛 프로세스가 포트를 잡고 있으면
# 새 서버는 뜨지 못하고 healthz는 옛 서버가 응답해 조용히 잘못된 상태가 된다.
for name in ('tunnel', 'server'):
    proc = globals().get(name)
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
        print('이전', name, '종료')

env = dict(os.environ, STT_MODEL=MODEL, STT_DEVICE='auto', STT_PRELOAD='1')
server = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'app.main:app', '--host', '127.0.0.1', '--port', str(PORT)],
    env=env,
    stdout=open('server.log', 'w'),
    stderr=subprocess.STDOUT,
)

started = False
for _ in range(90):
    if server.poll() is not None:
        break
    try:
        if requests.get(f'http://127.0.0.1:{PORT}/healthz', timeout=2).ok:
            started = True
            break
    except Exception:
        pass
    time.sleep(2)

if not started:
    raise RuntimeError('서버 시작 실패 (포트 ' + str(PORT) + ' 가 이미 사용 중일 수 있습니다)\n'
                       + open('server.log').read()[-3000:])

print('server:', requests.get(f'http://127.0.0.1:{PORT}/v1/info').json())

# 외부 접근용 임시 터널 (공개 URL이므로 민감한 데이터 업로드 금지)
PUBLIC_URL = None
CF = '/usr/local/bin/cloudflared'
if not os.path.exists(CF):
    subprocess.run([
        'wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', CF,
    ], check=True)
    os.chmod(CF, 0o755)

tunnel = subprocess.Popen(
    [CF, 'tunnel', '--url', f'http://127.0.0.1:{PORT}', '--no-autoupdate'],
    stdout=open('cloudflared.log', 'w'),
    stderr=subprocess.STDOUT,
)
for _ in range(45):
    match = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', open('cloudflared.log').read())
    if match:
        PUBLIC_URL = match.group(0)
        break
    time.sleep(1)

print('PUBLIC_URL:', PUBLIC_URL or '(터널 생성 실패 - 로컬에서만 사용)')


## 테스트 1 — 배치 전사 (파일 업로드 → 폴링 → 결과)

`jfk.wav`(영어 음성)를 업로드하고 `GET /v1/transcriptions/{id}`로 상태를 폴링한 뒤,
`/result`에서 텍스트와 SRT를 받아 출력한다.

- 기대: `status: completed` → `TEXT: And so my fellow Americans...`
- 실패하면 `error`의 `code`/`message`를 출력한다.
  - `dependency_missing`: 서버에 ffmpeg가 없음 → 셀 3 재실행
  - `invalid_audio`: 입력 파일이 손상/무음
  - `internal` + `cublas`: CUDA 12 라이브러리 문제 → 셀 3 재실행 후 셀 5 재기동
- 배치 경로: 업로드 → ffmpeg로 16 kHz mono WAV 변환 → faster-whisper 전사


In [ ]:
import os, time, requests

API = f'http://127.0.0.1:{PORT}'
WAV = 'jfk.wav'
if not os.path.exists(WAV) or os.path.getsize(WAV) < 1024:
    !wget -q -O jfk.wav https://github.com/ggerganov/whisper.cpp/raw/master/samples/jfk.wav
print('wav:', os.path.getsize(WAV), 'bytes')

with open(WAV, 'rb') as fh:
    res = requests.post(
        f'{API}/v1/transcriptions',
        files={'file': ('jfk.wav', fh, 'audio/wav')},
        data={'language': 'en'},
    )
print(res.status_code, res.text[:400])
if res.status_code != 202:
    raise RuntimeError('잡 생성 실패')

job_id = res.json()['id']

st = {}
for _ in range(180):
    st = requests.get(f'{API}/v1/transcriptions/{job_id}').json()
    print('status:', st['status'], '| progress:', st['progress'], '| error:', st.get('error'))
    if st['status'] in {'completed', 'failed', 'canceled'}:
        break
    time.sleep(1)

if st.get('status') != 'completed':
    err = st.get('error') or {}
    message = str(err.get('message') or '')
    print('전사 실패:', err.get('code'), '-', message)
    if 'metadata_errors' in message:
        print('PyAV 버전 충돌입니다. 셀 3을 다시 실행해 av를 19 미만으로 내린 뒤 셀 5를 재실행하세요.')
    if 'cublas' in message.lower() or 'cudnn' in message.lower():
        print('CUDA 12 라이브러리 문제입니다. 셀 3을 다시 실행한 뒤 셀 5(서버)를 재기동하세요.')
        print('셀 3만 다시 실행하면 이미 뜬 서버는 옛 환경 그대로라 같은 오류가 반복됩니다.')
    raise RuntimeError('전사 실패: ' + (message or str(st.get('status'))))

r = requests.get(f'{API}/v1/transcriptions/{job_id}/result')
print('TEXT:', r.json()['text'])
print(requests.get(f'{API}/v1/transcriptions/{job_id}/result', params={'format': 'srt'}).text)


## 테스트 2 — 실시간 스트리밍 (WebSocket)

같은 `jfk.wav`를 ffmpeg로 PCM(16 kHz, mono, s16le)으로 풀어 100 ms 프레임으로
`WS /v1/stream`에 보내고, 서버가 보내는 이벤트를 그대로 출력한다.

- 기대 순서: `ready` → `partial`(여러 번) → `final` → `done`
- `partial`은 미확정 가설, `final`은 확정 세그먼트다. 종료는 JSON `{"type":"end"}`.
- 모델 추론은 배치와 공유되므로, 배치가 도는 중이면 이벤트가 조금 늦게 올 수 있다.


In [ ]:
import asyncio, json, subprocess, websockets

def to_pcm(path):
    return subprocess.run(
        ['ffmpeg', '-v', 'error', '-i', path, '-f', 's16le', '-ac', '1', '-ar', '16000', '-'],
        check=True, capture_output=True,
    ).stdout

async def stream_demo():
    pcm = to_pcm('jfk.wav')
    step = 3200  # 100 ms, 16-bit mono
    async with websockets.connect(f'ws://127.0.0.1:{PORT}/v1/stream?language=en') as ws:
        print(json.loads(await ws.recv()))  # ready

        async def sender():
            for i in range(0, len(pcm), step):
                await ws.send(pcm[i:i + step])
                await asyncio.sleep(0.1)  # 실시간 속도
            await ws.send(json.dumps({'type': 'end'}))

        async def receiver():
            async for msg in ws:
                ev = json.loads(msg)
                if ev['type'] in {'partial', 'final'}:
                    print(f"[{ev['type']:7}] {ev['text']}")
                elif ev['type'] == 'done':
                    print('done:', ev)
                    break
                elif ev['type'] == 'error':
                    print('error:', ev)

        await asyncio.gather(sender(), receiver())

await stream_demo()

## 테스트 3 — 동시 요청 (배치 2건 + 스트리밍 1건)

배치 잡 2개를 연달아 접수하고 **동시에** 스트리밍 세션 1개를 열어, 셋 다 끝까지 가는지 확인한다.
"실시간과 일괄, 동시에 처리" 요구를 확인하는 셀이다. (먼저 테스트 1로 `jfk.wav`를 준비해 둔다)

- 확인: 두 배치 모두 `completed` + `error == null`, 스트림은 `final` 1개 이상 + `done` 도달 + `error` 없음
- 주의: 접수·수용·격리는 동시에 되지만 **추론은 `_infer_lock`으로 직렬화**되어 있다(단일 GPU 전제).
  따라서 이 셀은 "서로 간섭 없이 전부 성공하는가"를 보는 것이고, 처리량 측정이 아니다.
- 실패하면 각 잡의 상태/이벤트를 그대로 출력하고 예외로 멈춘다.


In [ ]:
import asyncio, json, subprocess, time, requests, websockets

API = f'http://127.0.0.1:{PORT}'
WAV = 'jfk.wav'

def submit_batch(name):
    with open(WAV, 'rb') as fh:
        res = requests.post(
            f'{API}/v1/transcriptions',
            files={'file': (name, fh, 'audio/wav')},
            data={'language': 'en'},
        )
    if res.status_code != 202:
        raise RuntimeError(f'잡 생성 실패: {res.status_code} {res.text[:200]}')
    return res.json()['id']

def to_pcm(path):
    return subprocess.run(
        ['ffmpeg', '-v', 'error', '-i', path, '-f', 's16le', '-ac', '1', '-ar', '16000', '-'],
        check=True, capture_output=True,
    ).stdout

async def stream_once(events):
    pcm = to_pcm(WAV)
    step = 3200  # 100 ms
    async with websockets.connect(f'ws://127.0.0.1:{PORT}/v1/stream?language=en') as ws:
        events.append(json.loads(await ws.recv()))  # ready

        async def sender():
            for i in range(0, len(pcm), step):
                await ws.send(pcm[i:i + step])
                await asyncio.sleep(0.05)  # 배치와 겹치도록 실시간의 2배 속도
            await ws.send(json.dumps({'type': 'end'}))

        async def receiver():
            async for msg in ws:
                ev = json.loads(msg)
                events.append(ev)
                if ev['type'] == 'done':
                    break

        await asyncio.gather(sender(), receiver())

async def wait_all(ids):
    pending, results = set(ids), {}
    while pending:
        for jid in list(pending):
            st = requests.get(f'{API}/v1/transcriptions/{jid}').json()
            if st['status'] in {'completed', 'failed', 'canceled'}:
                results[jid] = st
                pending.discard(jid)
        if pending:
            await asyncio.sleep(0.5)
    return results

batch_ids = [submit_batch('concurrent_a.wav'), submit_batch('concurrent_b.wav')]
print('동시 접수한 배치 잡:', batch_ids)

stream_events = []
started = time.time()
# 서버가 응답하지 않으면 셀이 영원히 매달리지 않도록 10분 상한을 둔다.
job_results, _ = await asyncio.wait_for(
    asyncio.gather(wait_all(batch_ids), stream_once(stream_events)), timeout=600
)
elapsed = time.time() - started

ok = True
for jid, st in job_results.items():
    ok = ok and st['status'] == 'completed' and st.get('error') is None
    print(f"{jid}: status={st['status']} progress={st['progress']} error={st.get('error')}")

kinds = [e['type'] for e in stream_events]
print('스트림 이벤트:', {k: kinds.count(k) for k in ('ready', 'partial', 'final', 'done', 'error')})
stream_ok = 'done' in kinds and 'final' in kinds and 'error' not in kinds

if not ok or not stream_ok:
    raise RuntimeError('동시 처리 검증 실패 - 위 상태/이벤트를 확인하세요.')
print(f'동시 처리 완료: 배치 {len(job_results)}건 + 스트리밍 1세션, {elapsed:.1f}s')


In [ ]:
print(open('server.log').read()[-2000:])

# 정리
# server.terminate(); tunnel.terminate()

## 참고

- API 문서: 서버 실행 후 `/docs`
- 모델 변경: `MODEL` 값을 바꾸고 프리로드/서버 셀을 다시 실행
- 인증: `STT_API_KEY=...` 환경 변수를 주면 HTTP는 `Authorization: Bearer`, WS는 `?api_key=`가 필요
- 세션 한도: `STT_MAX_ACTIVE_STREAMS`, `STT_MAX_ACTIVE_JOBS`
- Colab은 세션/유휴 시간 제한이 있고 영구 운영에 적합하지 않다. 운영은 `./run.sh` 사용.